In [ ]:
# Dependencies added for this notebook:
#   torchsynth==1.0.2            (modular synth components for §3.1)
#   pytorch-lightning==1.9.5     (torchsynth API compatibility)
#   setuptools<70                (restores pkg_resources used by torchsynth)
#
# To install into the conda env:
#   pip install torchsynth "setuptools<70" "pytorch-lightning<2.0"

import sys, os, time, copy, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import ipywidgets as widgets

from src.ddsp import DDSPAutoencoder, MultiScaleSpectralLoss, InharmonicityLoss
from src.training_divergence import (
    AnimalSynth, fit_synth_to_target, load_esc50_animals,
    load_audio_chunks, make_probe_trajectory,
    finetune_step, save_divergent_checkpoint,
)
from src.training_divergence.finetune_utils import save_inharmonic_checkpoint
from src.ddsp.dataset import URMPViolinDataset
from torch.utils.data import DataLoader

SAMPLE_RATE = 16000
HOP_LENGTH  = 64
MODELS_DIR  = "../models"
BASELINE_CKPT = os.path.join(MODELS_DIR, "ddsp_baseline_violin.pt")


def load_baseline(freeze=True):
    ckpt = torch.load(BASELINE_CKPT, map_location='cpu', weights_only=False)
    model = DDSPAutoencoder(**ckpt['config'])
    model.load_state_dict(ckpt['model_state_dict'])
    if freeze:
        model.eval()
        for p in model.parameters():
            p.requires_grad_(False)
    return model, ckpt['config']


def specshow(audio, sr=16000, title='', ax=None, fmax=8000.0):
    import librosa, librosa.display
    wav = audio.squeeze().detach().cpu().numpy()
    own = ax is None
    if own:
        _, ax = plt.subplots(figsize=(8, 2.5))
    D = librosa.amplitude_to_db(
        np.abs(librosa.stft(wav, n_fft=1024, hop_length=256)), ref=np.max)
    librosa.display.specshow(D, sr=sr, hop_length=256, x_axis='time', y_axis='log',
                              ax=ax, fmax=fmax)
    ax.set_title(title)
    if own:
        plt.tight_layout()
        plt.show()


def play(audio, sr=16000, normalize=True):
    wav = audio.squeeze().detach().cpu().numpy()
    if normalize and np.abs(wav).max() > 1e-8:
        wav = wav / np.abs(wav).max() * 0.9
    display(Audio(wav, rate=sr))


print("Setup complete.  Baseline checkpoint exists:", os.path.exists(BASELINE_CKPT))


# Notebook 3: Training-Time Active Divergence

*Class: Active Divergence with Generative Deep Learning*

---

In Notebook 2 we modified a **frozen** model at inference time — bending its outputs without
changing its weights. This notebook goes deeper: we **change training itself** to deliberately
introduce divergence.

Broad et al. identify three major training-time mechanisms:

| Mechanism | Description |
|-----------|-------------|
| **Inspiring set** | Target distribution chosen to be *outside* what the synthesizer can faithfully reproduce — approximation creates novelty |
| **Domain drift / fine-tuning** | Continue training on foreign audio; model drifts toward a hybrid timbre |
| **Loss modification** | Rewarding properties the original training signal actively discourages |

All three produce **irreversible** changes to the model's internal world — new weights, not just
new inputs.

### Three techniques in this notebook
1. **§3.1 Inspiring Set** — Hagiwara et al.: optimise a limited modular synth against animal
   vocalizations it can never reproduce
2. **§3.2 Divergent Fine-tuning** — Continue training the DDSP violin model on EMF electrical
   noise; hear it drift toward a hybrid violin-buzz
3. **§3.3 Loss Hacking** — Add an inharmonicity term to the training objective; watch the violin
   model drift toward metallic, bell-like tones


---
## §3.1  Inspiring Set: Animal Vocalizations via a Limited Synthesizer

### Background — Hagiwara et al. (2022)

Hagiwara et al. synthesize animal vocalizations with a **differentiable modular synthesizer**.
The crucial design choice is that the synthesizer is *intentionally limited*:

> "Because the synthesizer cannot faithfully reproduce the target, gradient descent finds
> parameters that echo the target's most prominent acoustic feature while filling the rest
> with synthesizer artefacts."

This is the **inspiring-set** concept (Broad et al.): the creative output emerges from the
*gap* between what the model can produce and what the target demands.  The synthesizer's
constraints are the artistic instrument.

### Our synth (`AnimalSynth`)

```
pitch trajectory  (16 linear-interpolated MIDI control points)
        ↓
FM carrier (SineVCO-style) ─── amplitude envelope ─── mix ─── audio
        ↑                                               ↑
FM modulator (depth + rate)                      white noise floor
```

**Deliberate limitations that create divergence:**
- 16 pitch control points cannot capture rapid bird trills or insect chirp patterns
- Single-operator FM cannot reproduce multi-formant frog calls
- White noise floor is not shaped like breath, wing, or rain noise

We optimise per sample with a multi-scale spectral loss for 300 iterations (~3 s on CPU per
sample).  The result echoes the target without copying it.

### Data

ESC-50 animal categories: `chirping_birds`, `frog`, `crickets`, `crow`, `insects`


In [ ]:
ESC50_DIR = "/mnt/mariadata/datasets/ESC-50"

print("Loading animal vocalizations from ESC-50 ...")
animals = load_esc50_animals(
    ESC50_DIR,
    categories={"chirping_birds", "frog", "crickets", "crow", "insects"},
    max_per_category=2,
    duration=2.0,
    sample_rate=SAMPLE_RATE,
)
print(f"Loaded {len(animals)} clips:")
for a in animals:
    print(f"  {a['category']:20s}  {a['filename']}")


### Optimization — per-sample fitting

300 Adam steps per clip.  Loss = multi-scale spectral distance (target vs. synth output).
Watch the spectrogram comparison: the synth captures the *average* pitch contour and envelope
shape but loses the biological texture.  That loss is the creative gain.


In [ ]:
fitted = []

for item in animals:
    cat = item['category']
    print(f"\n── Fitting: {cat} ({item['filename']}) ──")
    synth, losses = fit_synth_to_target(
        item['audio'],
        n_iter=300,
        lr=0.05,
        n_ctrl=16,
        sample_rate=SAMPLE_RATE,
        verbose=True,
    )
    with torch.no_grad():
        approx = synth()

    params = synth.get_param_dict()
    print(f"  f0≈{np.mean(params['pitch_ctrl_midi']):.1f} MIDI  "
          f"fm_depth={params['fm_depth']:.2f}  "
          f"fm_rate={params['fm_rate_hz']:.1f}Hz  "
          f"noise={params['noise_level_gain']:.3f}")

    fitted.append({
        'category': cat,
        'filename': item['filename'],
        'target': item['audio'],
        'approx': approx,
        'synth': synth,
        'losses': losses,
        'params': params,
    })

print("\nAll fits complete.")


In [ ]:
# ── Spectrogram comparison + loss curves ─────────────────────────────────────
n = len(fitted)
fig, axes = plt.subplots(n, 3, figsize=(14, 2.8 * n))
if n == 1:
    axes = axes[None, :]

for row, item in enumerate(fitted):
    ax_t, ax_a, ax_l = axes[row]
    specshow(item['target'], title=f"{item['category']} — TARGET",        ax=ax_t)
    specshow(item['approx'], title=f"{item['category']} — SYNTH APPROX",  ax=ax_a)
    ax_l.plot(item['losses'], color='steelblue', lw=1.5)
    ax_l.set(xlabel='Iteration', ylabel='Spectral loss', title='Optimization loss')
    ax_l.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ── Audio playback ────────────────────────────────────────────────────────────
for item in fitted:
    print(f"\n{'─'*55}")
    print(f"Category: {item['category']}  |  {item['filename']}")
    print("  Target:")
    play(item['target'])
    print("  Synth approximation:")
    play(item['approx'])


### Interactive parameter exploration

Sliders perturb the fitted parameters of one clip.  Explore the creative neighbourhood around
the optimised solution: nearby configurations sound *related* to both the target and the direct
fit, but are neither.  This is the parameter space of the inspiring-set approximation.


In [ ]:
if not fitted:
    print("Run the optimization cells above first.")
else:
    clip_opts = [(f"{r['category']} ({r['filename']})", i) for i, r in enumerate(fitted)]

    sel     = widgets.Dropdown(options=clip_opts, description='Clip:',
                               style={'description_width': '50px'})
    pitch_s = widgets.FloatSlider(value=0.0,  min=-12.0, max=12.0, step=0.5,
                                   description='Pitch shift (semitones)',
                                   style={'description_width': '200px'})
    fm_d_s  = widgets.FloatSlider(value=0.0,  min=-3.0,  max=6.0,  step=0.1,
                                   description='FM depth add',
                                   style={'description_width': '200px'})
    fm_r_s  = widgets.FloatSlider(value=1.0,  min=0.1,   max=5.0,  step=0.05,
                                   description='FM rate multiplier',
                                   style={'description_width': '200px'})
    noise_s = widgets.FloatSlider(value=0.0,  min=-3.0,  max=3.0,  step=0.1,
                                   description='Noise level add (log)',
                                   style={'description_width': '200px'})
    btn     = widgets.Button(description='▶  Synthesise & Play',
                              button_style='primary',
                              layout=widgets.Layout(width='200px'))
    out_w   = widgets.Output()

    def on_synth(b):
        with out_w:
            out_w.clear_output(wait=True)
            idx = sel.value
            base = fitted[idx]['synth']
            perturbed = base.perturb(
                pitch_shift=pitch_s.value,
                fm_depth_add=fm_d_s.value,
                fm_rate_mult=fm_r_s.value,
                noise_gain_add=noise_s.value,
            )
            with torch.no_grad():
                audio = perturbed()
            fig, axes = plt.subplots(1, 2, figsize=(12, 2.5))
            specshow(fitted[idx]['approx'], title='Fitted (reference)', ax=axes[0])
            specshow(audio,                 title='Perturbed',          ax=axes[1])
            plt.tight_layout()
            plt.show()
            play(audio)

    btn.on_click(on_synth)
    display(widgets.VBox([sel, pitch_s, fm_d_s, fm_r_s, noise_s, btn, out_w]))


---
## §3.2  Divergent Fine-tuning: Violin → EMF Noise

### Concept: "Find a Fertile Maladaptation"

Fine-tuning a model on foreign data does not simply teach it new content — it *distorts* the
model's existing representations.  When fine-tuning stops **before convergence** (deliberately),
the model exists in a liminal zone:

- It still carries the violin's harmonic vocabulary
- But it is being pulled toward the timbre of the new source
- The in-between zone — under-adapted, partially deformed — is where the most interesting sounds live

This is the training-time version of the "interesting region" from Broad et al.: the creative
potential sits at the **edge of maladaptation**, not at convergence.

### Data: EMF electrical noise

We fine-tune on **electromagnetic interference recordings** — 50 Hz hum and interference
patterns from `/mnt/mariadata/datasets/emf-noises-freesound/`.  These are maximally different
from violin: no pitch, harmonic structure from power-grid frequencies (50/100/150 Hz), dense noise floor.

Because CREPE extracts unreliable pitch from noise (periodicity ≈ 0.001), we assign a
**fixed f0 = 100 Hz** to all fine-tuning chunks.  This is pedagogically motivated: we tell the
model *"at 100 Hz, produce the timbre you hear in the training data"* — and that timbre is
electrical buzz.

### Training loop

The loop below is fully visible and modifiable.  After every `SAVE_EVERY` steps it:
1. Saves a checkpoint to `models/ddsp_divergent_violin_emf_step<N>.pt`
2. Renders the same fixed probe trajectory through the current model

Compare probe audio across steps to hear divergence accumulate.

---


In [ ]:
# ── Set the fine-tuning audio directory ──────────────────────────────────────
# Default: EMF noise recordings.
# To experiment: point to any directory of .wav files (e.g. a flute from URMP).

FINETUNE_DIR_DEFAULT = "/mnt/mariadata/datasets/emf-noises-freesound"
SOURCE_TAG = "violin"
TARGET_TAG = "emf"

dir_widget = widgets.Text(
    value=FINETUNE_DIR_DEFAULT,
    description='Fine-tune dir:',
    layout=widgets.Layout(width='700px'),
    style={'description_width': '120px'},
)
display(dir_widget)
print("Edit the path above if needed, then run the next cell.")


In [ ]:
# ── Hyperparameters (all CPU-feasible) ───────────────────────────────────────
FINETUNE_DIR   = dir_widget.value
CHUNK_DURATION = 2.0      # seconds per training clip
FIXED_F0_HZ    = 100.0   # constant f0 for unpitched EMF source
MAX_CHUNKS     = 20       # dataset cap — keeps epoch time short
BATCH_SIZE     = 2        # clips per gradient step
N_EPOCHS       = 5        # passes through data
LR             = 3e-4     # Adam learning rate
SAVE_EVERY     = 10       # steps between checkpoint + probe render
FFT_SIZES_FT   = (2048, 1024, 512, 256)

PROBE_F0_HZ       = 220.0   # A3 — violin's natural range
PROBE_LOUDNESS_DB = -30.0
PROBE_DURATION_S  = 2.0

print(f"Loading chunks from: {FINETUNE_DIR}")
chunks = load_audio_chunks(
    FINETUNE_DIR,
    chunk_duration=CHUNK_DURATION,
    sample_rate=SAMPLE_RATE,
    max_chunks=MAX_CHUNKS,
    fixed_f0_hz=FIXED_F0_HZ,
    hop_length=HOP_LENGTH,
)
print(f"Loaded {len(chunks)} chunks × {CHUNK_DURATION}s")

# ── Model + optimiser ─────────────────────────────────────────────────────────
ft_model, ft_config = load_baseline(freeze=False)
ft_model.train()
ft_optimizer = torch.optim.Adam(ft_model.parameters(), lr=LR)
ft_criterion = MultiScaleSpectralLoss(fft_sizes=FFT_SIZES_FT)

# ── Fixed probe trajectory ────────────────────────────────────────────────────
probe_f0, probe_loud = make_probe_trajectory(
    f0_hz=PROBE_F0_HZ, loudness_db=PROBE_LOUDNESS_DB,
    duration=PROBE_DURATION_S, sample_rate=SAMPLE_RATE, hop_length=HOP_LENGTH,
)
print(f"Model params: {sum(p.numel() for p in ft_model.parameters()):,}")
print("Ready.")


### Fine-tuning loop (notebook-side)

Read this code.  You can pause between epochs to listen to the probe, adjust `LR` or
`N_EPOCHS`, and re-run.


In [ ]:
ft_probe_audios  = []   # list of (label, tensor) for playback + strip
ft_loss_history  = []   # [(step, loss_value)]
ft_step = 0


def _sample_batch(chunk_list, bs):
    idxs = torch.randperm(len(chunk_list))[:bs].tolist()
    n_min = min(chunk_list[i]['f0_hz'].shape[0] for i in idxs)
    return {
        'audio':    torch.stack([chunk_list[i]['audio']              for i in idxs]),
        'f0_hz':    torch.stack([chunk_list[i]['f0_hz'][:n_min]     for i in idxs]),
        'loudness': torch.stack([chunk_list[i]['loudness'][:n_min]  for i in idxs]),
    }


@torch.no_grad()
def render_probe(model):
    model.eval()
    audio = model(probe_f0, probe_loud)['audio'].squeeze()
    model.train()
    return audio


# ── Baseline probe (step 0) ───────────────────────────────────────────────────
ft_probe_audios.append(('step 0 (baseline)', render_probe(ft_model)))
print(f"Starting fine-tuning: {N_EPOCHS} epochs, {len(chunks) // BATCH_SIZE} steps/epoch\n")

t_start = time.time()

for epoch in range(N_EPOCHS):
    steps_this_epoch = len(chunks) // BATCH_SIZE
    epoch_loss_sum = 0.0

    for _ in range(steps_this_epoch):
        batch = _sample_batch(chunks, BATCH_SIZE)
        loss_val = finetune_step(ft_model, ft_optimizer, ft_criterion, batch)
        ft_loss_history.append((ft_step, loss_val))
        epoch_loss_sum += loss_val
        ft_step += 1

        if ft_step % SAVE_EVERY == 0:
            pa = render_probe(ft_model)
            label = f'step {ft_step}'
            ft_probe_audios.append((label, pa))
            ckpt = save_divergent_checkpoint(
                ft_model, ft_step, SOURCE_TAG, TARGET_TAG, MODELS_DIR, ft_config)
            print(f"  ✓ {os.path.basename(ckpt)}")

    avg = epoch_loss_sum / steps_this_epoch
    print(f"Epoch {epoch + 1}/{N_EPOCHS}  avg_loss={avg:.4f}  "
          f"elapsed={time.time() - t_start:.0f}s")

print(f"\nFine-tuning complete. Total steps: {ft_step}")


In [ ]:
# ── Loss curve ────────────────────────────────────────────────────────────────
if ft_loss_history:
    steps, vals = zip(*ft_loss_history)
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(steps, vals, lw=1.2, color='crimson', alpha=0.8)
    ax.set(xlabel='Step', ylabel='Spectral loss',
           title='Fine-tuning loss — violin → EMF')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Probe spectrogram strip ───────────────────────────────────────────────────
n_p = len(ft_probe_audios)
if n_p:
    fig, axes = plt.subplots(1, n_p, figsize=(4 * n_p, 3), sharey=True)
    if n_p == 1:
        axes = [axes]
    for ax, (label, audio) in zip(axes, ft_probe_audios):
        specshow(audio, title=label, ax=ax)
    plt.suptitle('Probe output — fixed f0=220 Hz, fixed loudness', y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Listen to divergence accumulating ────────────────────────────────────────
for label, audio in ft_probe_audios:
    print(f"  {label}:")
    play(audio)


---
## §3.3  Loss Hacking: Inharmonicity Penalty

### Concept

Loss modification is the most direct form of training-time active divergence: we **rewrite the
objective function** itself.  The model trains on the same data with the same architecture —
but the thing it is being optimised *for* changes.

Here we add an **inharmonicity term** to the multi-scale spectral loss.  Rather than always
rewarding faithful reconstruction, we simultaneously reward *spreading harmonic energy across
more overtones*.

### Mathematical definition

The DDSP decoder outputs `harmonic_dist` — a softmax distribution
$\mathbf{p} \in \Delta^{K-1}$ over $K = 100$ harmonics.

A **concentrated** distribution (energy mainly in harmonics 1–3) sounds like a pure violin tone.
A **spread** distribution (equal energy across all 100 harmonics) sounds metallic, dense,
bell-like — precisely because no single harmonic dominates.

We maximise the **Shannon entropy** of this distribution:

$$H(\mathbf{p}) = -\sum_{k=1}^{K} p_k \log(p_k + \varepsilon)$$

The loss term is **negative entropy** (so minimising it maximises spread):

$$\mathcal{L}_{\text{inh}}(\mathbf{p}) = -H(\mathbf{p}) = \sum_{k=1}^{K} p_k \log(p_k + \varepsilon)$$

Combined objective:

$$\mathcal{L} = \mathcal{L}_{\text{spectral}}(\hat{x},\, x) + \lambda\,\mathcal{L}_{\text{inh}}(\mathbf{p})$$

**Why this is cheap:** `InharmonicityLoss` operates directly on the decoder's output tensor —
no FFT, no audio synthesis needed for the gradient path.

### Listening for the trade-off

As $\lambda$ increases:

| Observable | Direction |
|------------|-----------|
| Spectral reconstruction loss | **rises** — model stops tracking violin accurately |
| Inharmonicity loss (`−H`) | **falls** — harmonic distribution spreads |
| Probe audio | **drifts** from violin → metallic / bell-like / buzzy |

---


In [ ]:
inh_weight_widget = widgets.FloatSlider(
    value=5.0, min=0.0, max=30.0, step=0.5,
    description='λ (inharmonicity weight):',
    style={'description_width': '220px'},
    layout=widgets.Layout(width='600px'),
)
display(inh_weight_widget)
print("Adjust λ, then run the training loop cell.")
print("  λ = 0 :  pure reconstruction (same as baseline training)")
print("  λ = 5 :  mild inharmonicity — slightly metallic")
print("  λ ≥15 :  strong spread — noticeably bell/buzz-like")


In [ ]:
# ── §3.3 hyperparameters ─────────────────────────────────────────────────────
INH_WEIGHT        = inh_weight_widget.value
N_EPOCHS_INH      = 5
STEPS_PER_EPOCH_INH = 30   # ~10 s/epoch on CPU
BATCH_SIZE_INH    = 2
LR_INH            = 1e-4   # deliberately small — we want to see drift, not collapse
SAVE_EVERY_INH    = 10
FFT_SIZES_INH     = (2048, 1024, 512, 256)

INH_PROBE_F0   = 220.0
INH_PROBE_LOUD = -30.0
INH_PROBE_DUR  = 2.0

print(f"λ = {INH_WEIGHT}")

# ── Fresh model from baseline ─────────────────────────────────────────────────
inh_model, inh_config = load_baseline(freeze=False)
inh_model.train()
inh_optimizer  = torch.optim.Adam(inh_model.parameters(), lr=LR_INH)
inh_spec_crit  = MultiScaleSpectralLoss(fft_sizes=FFT_SIZES_INH)
inh_loss_fn    = InharmonicityLoss()

# ── Same violin dataset as baseline ──────────────────────────────────────────
PROCESSED_DIR = "../samples/processed"
violin_ds = URMPViolinDataset(PROCESSED_DIR)
violin_loader = DataLoader(violin_ds, batch_size=BATCH_SIZE_INH, shuffle=True,
                            collate_fn=URMPViolinDataset.collate_fn, drop_last=True)

# ── Probe ─────────────────────────────────────────────────────────────────────
inh_probe_f0, inh_probe_loud = make_probe_trajectory(
    f0_hz=INH_PROBE_F0, loudness_db=INH_PROBE_LOUD,
    duration=INH_PROBE_DUR, sample_rate=SAMPLE_RATE, hop_length=HOP_LENGTH,
)
print(f"Dataset: {len(violin_ds)} clips | Model params: {sum(p.numel() for p in inh_model.parameters()):,}")


### Training loop with inharmonicity loss

Two loss values are tracked separately at every step so you can see the trade-off directly.


In [ ]:
inh_spectral_hist = []   # [(step, val)]
inh_inh_hist      = []
inh_probe_audios  = []
inh_step = 0
data_iter = iter(violin_loader)


@torch.no_grad()
def render_inh_probe(model):
    model.eval()
    audio = model(inh_probe_f0, inh_probe_loud)['audio'].squeeze()
    model.train()
    return audio


# ── Baseline probe ────────────────────────────────────────────────────────────
inh_probe_audios.append(('λ=0 (baseline)', render_inh_probe(inh_model)))
print(f"Starting §3.3 training: λ={INH_WEIGHT}, {N_EPOCHS_INH} epochs\n")

t_start = time.time()

for epoch in range(N_EPOCHS_INH):
    for _ in range(STEPS_PER_EPOCH_INH):
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(violin_loader)
            batch = next(data_iter)

        inh_model.train()
        out = inh_model(batch['f0_hz'], batch['loudness'])

        s_loss = inh_spec_crit(out['audio'], batch['audio'])
        i_loss = inh_loss_fn(out['harmonic_dist'])
        total  = s_loss + INH_WEIGHT * i_loss

        inh_optimizer.zero_grad()
        total.backward()
        torch.nn.utils.clip_grad_norm_(inh_model.parameters(), 1.0)
        inh_optimizer.step()

        inh_spectral_hist.append((inh_step, s_loss.item()))
        inh_inh_hist.append((inh_step, i_loss.item()))
        inh_step += 1

        if inh_step % SAVE_EVERY_INH == 0:
            pa = render_inh_probe(inh_model)
            inh_probe_audios.append((f'step {inh_step}', pa))
            ckpt = save_inharmonic_checkpoint(inh_model, inh_step, INH_WEIGHT,
                                               MODELS_DIR, inh_config)
            print(f"  ✓ {os.path.basename(ckpt)}")

    avg_s = np.mean([v for _, v in inh_spectral_hist[-STEPS_PER_EPOCH_INH:]])
    avg_i = np.mean([v for _, v in inh_inh_hist[-STEPS_PER_EPOCH_INH:]])
    print(f"Epoch {epoch + 1}/{N_EPOCHS_INH}  "
          f"spectral={avg_s:.4f}  inh={avg_i:.4f}  "
          f"elapsed={time.time() - t_start:.0f}s")

print(f"\nTraining complete. Steps: {inh_step}")


In [ ]:
# ── Dual loss curves (twin y-axis) ────────────────────────────────────────────
if inh_spectral_hist and inh_inh_hist:
    ss, vs = zip(*inh_spectral_hist)
    si, vi = zip(*inh_inh_hist)

    fig, ax1 = plt.subplots(figsize=(11, 3.5))
    ax2 = ax1.twinx()

    l1, = ax1.plot(ss, vs, color='tomato',    lw=1.5, label='Spectral reconstruction loss ↑')
    l2, = ax2.plot(si, vi, color='steelblue', lw=1.5, label='Inharmonicity loss (−H) ↓', alpha=0.85)

    ax1.set(xlabel='Step', ylabel='Spectral loss',
            title=f'Loss hacking (λ={INH_WEIGHT}): reconstruction ↑ as inharmonicity ↓')
    ax2.set_ylabel('Inharmonicity loss (negative entropy)')
    ax1.grid(True, alpha=0.3)
    ax1.legend(handles=[l1, l2], loc='upper right')
    plt.tight_layout()
    plt.show()

    print("Spectral loss rises = model stops tracking the violin accurately.")
    print("Inharmonicity loss falls = harmonic distribution spreads across more overtones.")


In [ ]:
# ── Probe spectrogram strip ───────────────────────────────────────────────────
n_p = len(inh_probe_audios)
if n_p:
    fig, axes = plt.subplots(1, n_p, figsize=(4 * n_p, 3), sharey=True)
    if n_p == 1:
        axes = [axes]
    for ax, (label, audio) in zip(axes, inh_probe_audios):
        specshow(audio, title=label, ax=ax)
    plt.suptitle(f'Inharmonicity probe evolution (λ={INH_WEIGHT}, f0=220 Hz)', y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Listen to inharmonicity accumulating ─────────────────────────────────────
for label, audio in inh_probe_audios:
    print(f"  {label}:")
    play(audio)


---
## Summary: Three Modes of Training-Time Active Divergence

| Technique | What changes | Timbre trajectory |
|-----------|-------------|-------------------|
| **§3.1 Inspiring set** | Synth parameters per sample | Clean but echoing — biological texture replaced by FM artefact |
| **§3.2 Divergent fine-tuning** | Decoder weights → EMF domain | Hybrid violin-buzz, increasingly strange before convergence |
| **§3.3 Loss hacking** | Training objective → inharmonicity | Metallic, bell-like, spread-spectrum as reconstruction degrades |

**Common thread: controlled maladaptation.**  In each case the system is pushed toward something
it cannot fully reach.  The failure to arrive is where the interesting sounds live.

> "The best divergences come from constraints the artist did not put there on purpose."
> — paraphrase of the inspiring-set principle

---
*Active Divergence Class — Notebook 3 of 4*
